# 13회차 실습 노트북 — 데이터 기반 핀테크 서비스 기획 — 로보어드바이저

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adminmrobo/fintech_data_analysis/blob/main/notebooks/13_robo_advisor.ipynb)

해설 페이지: [https://adminmrobo.github.io/fintech_data_analysis/13_robo_advisor.html](https://adminmrobo.github.io/fintech_data_analysis/13_robo_advisor.html)

위에서부터 셀을 차례로 실행하세요(Shift+Enter). 데이터는 저장소 `data/` 폴더(fintech-data 복사본)에서 읽으며, 무작위 시드를 42로 고정해 해설 페이지의 대결 숫자와 같은 값이 나옵니다.

글 · 안상선 ((주)M-Robo 대표)

In [1]:
# 준비: 라이브러리와 데이터 불러오기 (Colab에서는 그대로 실행하면 됩니다)
import os, urllib.parse, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

BASE = 'https://raw.githubusercontent.com/adminmrobo/fintech_data_analysis/main/data/'
def load(path, **kw):
    """저장소의 data/ 폴더에서 CSV를 읽는다. 내 컴퓨터에 저장소가 있으면 그 파일을, 없으면 GitHub에서 읽는다."""
    for root in ('data', '../data'):
        f = os.path.join(root, path)
        if os.path.exists(f):
            return pd.read_csv(f, **kw)
    return pd.read_csv(BASE + urllib.parse.quote(path), **kw)

RS = 42   # 해설 페이지와 같은 숫자가 나오도록 무작위 시드를 고정

> 교육용 백테스트이며 투자 자문이 아닙니다. 거래 비용 0.1%를 가정했습니다.

## 1. 이동평균 교차 전략 백테스트

In [2]:
p = load('Revolution3/portfolio_data.csv')
def ma_bt(series, s, l, fee=0.001):
    x = pd.Series(np.asarray(series, float)); r = x.pct_change().fillna(0)
    sig = (x.rolling(s).mean() > x.rolling(l).mean()).astype(int).shift(1).fillna(0)
    strat = sig * r - sig.diff().abs().fillna(0) * fee
    eq, bh = (1 + strat).cumprod(), (1 + r).cumprod()
    mdd = lambda e: (e / e.cummax() - 1).min()
    return eq, bh, dict(전략배수=round(float(eq.iloc[-1]), 2), 보유배수=round(float(bh.iloc[-1]), 2), 전략낙폭=round(float(mdd(eq)), 3), 보유낙폭=round(float(mdd(bh)), 3))
for c in ['AMZN', 'BTC']:
    print(c, ma_bt(p[c], 20, 60)[2])

AMZN {'전략배수': 3.77, '보유배수': 7.41, '전략낙폭': -0.287, '보유낙폭': -0.341}
BTC {'전략배수': 113.75, '보유배수': 77.02, '전략낙폭': -0.751, '보유낙폭': -0.91}


In [3]:
eq, bh, _ = ma_bt(p.BTC, 20, 60)
plt.plot(eq / eq.cummax() - 1, label='MA 20/60'); plt.plot(bh / bh.cummax() - 1, c='gray', label='buy & hold')
plt.title('BTC drawdown'); plt.legend(); plt.show()

## 2. 위험 성향별 배분 (주식 3종 동일 비중 + 안전자산 연 2% 가정)

In [4]:
ret = p[['AMZN', 'DPZ', 'NFLX']].pct_change().dropna().mean(axis=1)
for w in [0, .2, .4, .6, .8, 1]:
    rr = w * ret + (1 - w) * 0.02 / 252; e = (1 + rr).cumprod()
    print(f'주식 {w:.0%}: 연 수익 {rr.mean() * 252:.1%}  연 변동성 {rr.std() * np.sqrt(252):.1%}  최대 낙폭 {(e / e.cummax() - 1).min():.1%}')

주식 0%: 연 수익 2.0%  연 변동성 0.0%  최대 낙폭 0.0%
주식 20%: 연 수익 9.5%  연 변동성 4.9%  최대 낙폭 -6.3%
주식 40%: 연 수익 16.9%  연 변동성 9.9%  최대 낙폭 -12.8%
주식 60%: 연 수익 24.4%  연 변동성 14.8%  최대 낙폭 -19.0%
주식 80%: 연 수익 31.9%  연 변동성 19.7%  최대 낙폭 -24.9%
주식 100%: 연 수익 39.3%  연 변동성 24.7%  최대 낙폭 -30.5%


**AI 판단 연습** — 같은 전략이 아마존에서는 지고 비트코인에서는 이겼습니다. "이 전략이 좋다"고 말하려면 무엇을 더 확인해야 할까요? (기간, 종목, 비용, 과최적화)